In [76]:
import os 
import boto3 
import yaml

# get aws s3 credentials from environment (not working in jupiter notebook)
# aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
# aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')

searching_file = 'credentials.yml'
lookup_folder=r'C:\MySpace\Working\DE01\data_engineering_practices'


def lookup_file_path (searching_file, lookup_folder):
    for root, dirs, files in os.walk(lookup_folder):
        for name in files:
            if name == searching_file:  
                return os.path.abspath(os.path.join(root, name))
            

# read credentials stored in yaml file into data dictionary
config_file_path = lookup_file_path(searching_file,lookup_folder)
with open(config_file_path, 'r') as f:
    try:
        config_data = yaml.safe_load(f)
        # print(config_data)
    except yaml.YAMLError as exc:
        print(exc)
# get Credentials
aws_access_key_id = config_data['aws']['aws_access_key_id']
aws_secret_access_key = config_data['aws']['aws_secret_access_key']




In [81]:
# Create aws session & s3 client
# create a authorized session using boto3 + credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)

# create s3 client to interact with AWS S3 (boto3 will automatically use configured credentials)
s3 = session.client(service_name='s3')

# get current date in timezone
import pytz
from datetime import datetime
today = datetime.now(tz=pytz.timezone('Australia/Sydney'))
year, month, day = today.strftime('%Y'), today.strftime('%m'), today.strftime('%d')


# Define destination bucket name and upload file folder
S3_DES_BUCKET_NAME = 'cm-aws-s3-destination'
 


SA Metro GTFS
- Loading Mode: Initial Full Load + Incremental Load
    - Initial Full Load: Load the previous 10 versions of GTFS
    - Incremental Load:
        - Get the latest version in `landing zone` & latest version available from API.
        - Execute load if the API's latest version has not been ingested.  
- Partitioned Destination: `<landing_bucket>/gtfs/<FEED VERSION>`

In [78]:
base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"
latest_version_feed = "static/latest/google_transit.zip"

import requests #python package to work with API
import os
from io import BytesIO
from zipfile import ZipFile
import datetime

# Official page: https://gtfs.adelaidemetro.com.au/
# URLs
base_url = "http://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"
latest_version_feed = "static/latest/google_transit.zip"

# binary data as response
request_headers={"Content-Type": "application/octet-stream"} #octet-stream: Data transmitted over networks or read from files in this format is processed as a stream of bytes. 

# Feed template
# practice: how to use string template + format
feed_url_template = "static/{version}/google_transit.zip"

# output
dest_path = 'destination'
os.makedirs(dest_path, exist_ok=True)

In [ ]:
# FULL LOAD
# 1. read data from SA Metro GTFS feed
# 2. upload feed version into landing bucket

# load FEED_VERSION from 1600 to 1610 from SA Metro GTFS
# 1. For loop the versions
# 2. Generate versioned URL
# 3. Request and handle response

print(f'Downloading multiple feed versions (1600 to 1610) from {base_url} ... to S3 bucket')
# 2. Download feed version using request
for version in range(1600,1611):
    feed_version_url=f'{base_url}/static/{version}/google_transit.zip'
    response=requests.get(url=feed_version_url,headers=request_headers)
    s3_des_file_path=f'k1/Angie/gtfs/{version}/{version}_google_transit.zip'# The destination path within S3

# runtime when using chunk iteration for content takes much longer time to complete, unreccomemended approach
#     for chunk in response.iter_content(chunk_size=8192):
#     file_like_object = BytesIO(chunk)

    file_like_object = BytesIO(response.content)
    s3.upload_fileobj(file_like_object, S3_DES_BUCKET_NAME, s3_des_file_path)
    print(f'Download version {version} completed!')

In [ ]:
# INCREMENTAL LOAD
# 1. Get: API's latest version & AWS S3's versions
# 2. Compare & logging whether the latest version is already ingested
# 3. Execute the ingestion if not already ingested

# Practice: define script into tasks / functions
# e.g., Func1: Get API Latest version
# Func2: Get AWS S3 versions and compare if API latest version is already ingested -> True/False
# Func3: Execute the API latest version ingestion if Func2 return False


# 1. Find the latest version
latest_version_number_url=f'{base_url}/{latest_version_number}'
response=requests.get(url=latest_version_number_url)
latest_version=response.text

# destination paths
s3_des_latest_file_path=f'k1/Angie/gtfs/latest/{latest_version}_google_transit.zip'# The destination path within S3
s3_des_file_path=f'k1/Angie/gtfs/{latest_version}/{latest_version}_google_transit.zip'# The destination path within S3

# check if this version is already ingested to S3 bucket
is_ingested=False
from botocore.exceptions import ClientError
try:
        # The head_object operation retrieves metadata without the body
        s3.head_object(Bucket=S3_DES_BUCKET_NAME, Key=s3_des_file_path)
        is_ingested=True
except ClientError as e:
    # If a ClientError is raised, check if the error code is 404 (NoSuchKey)
    if e.response['Error']['Code'] == '404':
        is_ingested=False
    else:
        # Re-raise the exception if it's a different error (e.g., permissions)
        raise

# if not, upload latest version to S3 bucket
if not is_ingested:
    # 2. Download latest GTFS feed version using request
    print(f'Downloading latest verion {latest_version} from {base_url} ...')
    # Find the latest feed version
    latest_feed_version_url=f'{base_url}/{latest_version_feed}'
    response=requests.get(url=latest_feed_version_url,headers=request_headers)
    # upload latest file to S3 bucket
    file_like_object = BytesIO(response.content)
    s3.upload_fileobj(file_like_object, S3_DES_BUCKET_NAME, s3_des_file_path)
    s3.upload_fileobj(file_like_object, S3_DES_BUCKET_NAME, s3_des_latest_file_path)
    print(f'Download version {latest_version} completed!')

    # for chunk in response.iter_content(chunk_size=8192):
    #         file_like_object = BytesIO(chunk)
    #         s3.upload_fileobj(file_like_object, S3_DES_BUCKET_NAME, s3_des_file_path)
           
    print(f'Download latest version {latest_version} completed!')
